## LLaMA 3.2 3B — Local Pipeline
Aspect-Based Sentiment Analysis using Ollama.
Processes restaurant reviews and extracts structured sentiment scores
across 5 aspects: overall, food, service, price, ambiance.

**Requirements:** Ollama running locally with `llama3.2` model downloaded.

In [ ]:
import ollama
import pandas as pd
import time
from pydantic import BaseModel, Field
import json

In [ ]:
try:
    response = ollama.chat(
        model='llama3.2',
        messages=[
            {
                'role': 'user',
                'content': 'Are you ready to analyse some data? Answer yes or no.'
            }
        ]
    )
    print("Model Response:", response['message']['content'])
except Exception as e:
    print("Error:", e)
    print("⚠️ Make sure the Ollama app is running and you have downloaded llama3.2!")

In [ ]:
# Define the structure for a single aspect
class AspectScore(BaseModel):
    score: float | None = None
    quotes: list[str] = []

# Define the overall structure we expect the model to return
class ReviewAnalysis(BaseModel):
    overall: AspectScore = AspectScore()
    food: AspectScore = AspectScore()
    service: AspectScore = AspectScore()
    price: AspectScore = AspectScore()
    ambiance: AspectScore = AspectScore()

In [ ]:
# We dynamically inject the schema into the prompt
schema_instructions = ReviewAnalysis.model_json_schema()

SYSTEM_PROMPT = f"""
[Context]

You are an expert restaurant critic performing structured data analysis to identify performance issues.

[Task]

Your job is to perform Aspect-Based Sentiment Analysis on the restaurant reviews. You must evaluate the customers sentiment regarding five specific aspects: "overall", "food", "service", "price" and "ambiance".

[Constraints]

For each category (overall, food, service, price, ambiance), assign a score:
-1.0 (Strong Negative)
-0.5 (Negative)
 0.0 (Neutral / Mixed)
+0.5 (Positive)
+1.0 (Strong Positive)
null (Not Mentioned)

You can use every number in between. But NEVER use a number lower than -1.0 or higher than 1.0.

For each aspect, provide quotes: a short verbatim excerpt from the review that justifies the score. If an aspect is not mentioned, set both score and quote to null.

The "quotes" field must be either:
- a list of strings (even if only one quote is provided), or
- null

Only assign a score if the review explicitly mentions that aspect. Do NOT infer or assume sentiment that is not directly stated. The "overall" aspect is an exception — it reflects your holistic reading of the review and may not have a direct quote, in which case set quote to null.
The overall category is not allowed to be null or empty, always add your evalution there.

Pay close attention to sarcasm. If the tone contradicts the actual event being described, adjust the score accordingly.

[Format]

Output ONLY valid JSON that strictly matches this schema:
{json.dumps(schema_instructions, indent=2)}

[Few-Shot Examples]

Example 1:

Review: "The fried chicken taste great and the fries were very fresh and the customer service was good. I can't complain. I have not had fried chicken in Chicago that taste like that!"

Output: {{
  "overall": {{"score": 0.7, "quotes": ["I can't complain"]}},
  "food": {{"score": 1.0, "quotes": ["the fried chicken taste great", "I have not had fried chicken in Chicago that taste like that"]}},
  "service": {{"score": 0.5, "quotes": ["customer service was good"]}},
  "price": {{"score": null, "quotes": null}},
  "ambiance": {{"score": null, "quotes": null}}
}}

Example 2:

Review: "One of the worst meals I have ever eaten, I was told this was an iconic place to eat do not go there. The place was dirty the food was cold orders were wrong. Had to get our own silverware and refills on drinks. They said it's a thirty minute wait on fried chicken dinner so you expect hot juicy chicken wrong!! Cold dry chicken greens had no flavor cooked in water and no seasoning. the best thing-in this restaurant is the exit door save your money."

Output: {{
  "overall": {{"score": -1.0, "quotes": ["the best thing-in this restaurant is the exit door save your money"]}},
  "food": {{"score": -1.0, "quotes": ["One of the worst meals I have ever eaten", "Cold dry chicken greens had no flavor cooked in water and no seasoning"]}},
  "service": {{"score": -0.2, "quotes": ["orders were wrong"]}},
  "price": {{"score": null, "quotes": null}},
  "ambiance": {{"score": -0.7, "quotes": ["the place was dirty"]}}
}}

Example 3:

Review: "A group of 6 of us were visiting had we heard good things about Mother's.  We had the Ferdi, fried chicken, gumbo and a couple sides. Everyone enjoyed their meals. Good service, good value for the amount of food. Would go back again."

Output: {{
  "overall":  {{"score": 0.7, "quotes": ["Would go back again."]}},
  "food":     {{"score": 0.7, "quotes": ["Everyone enjoyed their meals"]}},
  "service":  {{"score": 0.7, "quotes": ["Good service"]}},
  "price":    {{"score": 0.7, "quotes": ["good value for the amount of food"]}},
  "ambiance": {{"score": null, "quotes": null}}
}}

Example 4:

Review: "The service here is unbelievable. Everyone just treated you like family here. braised greens and the fried chicken are both a must try here. At the end of the day, you are not allowed to tip anyone here."

Output: {{
  "overall": {{"score": 0.8, "quotes": null}},
  "food": {{"score": 0.7, "quotes": ["braised greens and the fried chicken are both a must try here"]}},
  "service": {{"score": 1.0, "quotes": ["The service here is unbelievable", "Everyone just treated you like family here"]}},
  "price": {{"score": null, "quotes": null}},
  "ambiance": {{"score": null, "quotes": null}}
}}

Example 5: Review: "Oh fantastic, waited an hour for lukewarm food. Really worth the trip."

Output: {{
  "overall": {{"score": -0.8, "quotes": null}},
  "food": {{"score": -1.0, "quotes": ["waited an hour for lukewarm food"]}},
  "service": {{"score": -0.5, "quotes": ["waited an hour"]}},
  "price": {{"score": null, "quotes": null}},
  "ambiance": {{"score": null, "quotes": null}}
}}

"""

In [ ]:
def analyse_review_local(review_text, review_id, retries=3):
    """
    Gets the raw ollama response. Retries on failure.
    Returns None only after all retries fail.
    """
    for attempt in range(retries):
        try:
            response = ollama.chat(
                model='llama3.2',
                messages=[
                    {'role': 'system', 'content': SYSTEM_PROMPT},
                    {'role': 'user', 'content': review_text}
                ],
                format=ReviewAnalysis.model_json_schema(),
                options={'temperature': 0}
            )

            # Convert the ChatResponse object to a standard dictionary
            if hasattr(response, 'model_dump'):
                return response.model_dump()
            else:
                return dict(response)

        except Exception as e:
            print(f"Attempt {attempt + 1} failed: {e}")
            if attempt < retries - 1:
                time.sleep(1)
            else:
                print(f"All {retries} attempts failed for review {review_id}.")
                return None

In [ ]:
# Define the expected structure for our technical metadata
class OllamaMetadata(BaseModel):
    # We map the raw Ollama keys to our preferred DataFrame column names
    model_name: str | None = Field(default=None, alias="model")
    created_at: str | None = None
    total_duration_ns: int | None = Field(default=None, alias="total_duration")
    load_duration_ns: int | None = Field(default=None, alias="load_duration")
    prompt_eval_duration_ns: int | None = Field(default=None, alias="prompt_eval_duration")
    generation_duration_ns: int | None = Field(default=None, alias="eval_duration")
    input_token_count: int | None = Field(default=None, alias="prompt_eval_count")
    output_token_count: int | None = Field(default=None, alias="eval_count")

In [ ]:
def parse_metadata(response, review_id):
    """
    Extracts technical metadata using Pydantic.
    Returns a structure with None values if keys are missing.
    """
    # 1. Instantly create an empty fallback structure
    fallback_data = OllamaMetadata().model_dump()
    fallback_data["review_id"] = review_id

    if response is None:
        return fallback_data

    try:
        # 2. Validate the dictionary returned by Ollama.
        # Pydantic automatically handles the aliases (renaming keys)!
        validated_meta = OllamaMetadata.model_validate(response)

        # 3. Convert back to dict and append the ID
        final_dict = validated_meta.model_dump()
        final_dict["review_id"] = review_id

        return final_dict

    except Exception as e:
        print(f"Error parsing metadata for {review_id}: {e}")
        return fallback_data

In [ ]:
def parse_sentiment(response, review_id):
    """
    Extracts the aspect scores using Pydantic validation.
    Returns a consistent structure with None values if parsing fails.
    """
    # 1. Let Pydantic instantly create our empty fallback structure
    fallback_data = ReviewAnalysis().model_dump()
    fallback_data["review_id"] = review_id

    if response is None:
        return fallback_data

    try:
        message = response.get('message', {})
        content_str = message.get('content', '{}')

        # 2. Validate the JSON string directly against our Pydantic model
        validated_data = ReviewAnalysis.model_validate_json(content_str)

        # 3. Convert back to dict and append the ID
        final_dict = validated_data.model_dump()
        final_dict["review_id"] = review_id

        return final_dict

    except Exception as e:
        # Pydantic will raise a ValidationError if the output is malformed
        print(f"Error parsing sentiment for review {review_id}: {e}")
        return fallback_data

In [ ]:
def process_restaurant_reviews(df):
    sentiment_records = []
    metadata_records = []

    for index, row in df.iterrows():
        r_id = row['review_id']
        text = row['text']

        # 1. Get raw response
        raw_response = analyse_review_local(text, r_id)

        # 2. Extract Data
        sentiment_data = parse_sentiment(raw_response, r_id)
        meta_data = parse_metadata(raw_response, r_id)

        sentiment_records.append(sentiment_data)
        metadata_records.append(meta_data)

    # --- Create DataFrames ---
    df_aspects = pd.json_normalize(sentiment_records, sep='_')
    if not df_aspects.empty:
        df_aspects = df_aspects.set_index('review_id')

    df_metadata = pd.DataFrame(metadata_records)
    if not df_metadata.empty:
        df_metadata = df_metadata.set_index('review_id')

    return df_aspects, df_metadata

In [ ]:
file_path = "../data/restaurant_reviews_sample.csv"

# Load the CSV into a DataFrame
data = pd.read_csv(file_path)

# Display the first 5 rows to verify
print(data.head())


In [ ]:
ollama_aspects, ollama_metadata = process_restaurant_reviews(data)

In [ ]:
ollama_aspects

In [ ]:
ollama_metadata

In [ ]:
ollama_aspects.to_csv('../data/ollama_aspects_llama_final.csv', index=False)
ollama_metadata.to_csv('../data/ollama_metadata_llama_final.csv', index=False)